In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
import pylupnt as pnt

In [ ]:
# ECC vs inclination
fig = plt.figure(figsize=(4, 2))

ecc = np.linspace(0, 1.0, 100)
inc = np.arccos(np.sqrt(3 / 5 * (1 - ecc * ecc)))
incd = np.rad2deg(inc)


plt.plot(incd, ecc)
plt.xlabel("Inclination", fontsize=14)
plt.ylabel("Eccentricity", fontsize=14)
plt.xlim([30, 80])
plt.ylim([0, 1])
plt.grid(True)

## Setup

In [ ]:
from src.odsim import ODSim
import os

# od configuraitons
et0 = pnt.convert_time(pnt.gregorian_to_time(2030, 1, 1, 12, 0, 0), pnt.UTC, pnt.TAI)

odconfig = {
    "et0": et0,
    "meas": ["gs"],
    "gs_ids": ["LEGS1_X", "LEGS2_X", "DSS35_X"],
    "init_pos_std": 100e-3,  # km
    "init_vel_std": 1e-3,  # km/s
    "sigma_range": 1e-3,  # m
    "sigma_rangerate": 0.1e-6,  # km/s
    "elev_mask_deg": 10,  # degrees
    "std_acc": 1e-7,  # km/s^2
    "predict_time": 2 * 3600,  # seconds
}

odsim = ODSim(odconfig)
print("Ground Station Locations (ECEF)")
print(odsim.pos_gs)

# dynamics setup
dyn = pnt.NBodyDynamics()
dyn.set_integrator(pnt.IntegratorType.RKF45)
dyn.set_integrator_params(pnt.IntegratorParams(max_iter=20, abstol=1e-10, reltol=1e-10))
dyn.add_body(pnt.Body.Moon(20, 20))
dyn.add_body(pnt.Body.Earth())
dyn.add_body(pnt.Body.Sun())
dyn.set_time_step(60)  # 10 seconds time step
dyn.set_frame(pnt.MOON_CI)
dyn.set_autodiff(True)

os.makedirs("figs/od", exist_ok=True)

### pre-compute stm and measurements

In [ ]:
from src.odsim import propagate_sats_with_stm, get_range_and_rate

# constants
n_sma = 13
n_incs = 13
n_omega = 25
n_orbit = 4  # number of orbit to simulate

# results
results = np.ones((n_sma, n_incs, n_omega, 8)) * np.nan

stats_ratio = (n_orbit - 1) / n_orbit  # ratio of the stats period

# initial conditions
smas = np.linspace(4000, 16000, n_sma) * 1e3  # [km] -> [m]
incs = np.deg2rad(np.linspace(40, 70, n_incs))
omegas = np.linspace(0, 2 * np.pi, n_omega)

In [ ]:
from src.odsim import gridsearch_od_parallel
import os

filename = "data/gridsearch_od/gridsearch_od_results.npy"

if os.path.exists(filename):
    results = np.load(filename)
    print("Loaded existing results from", filename)
else:
    results = gridsearch_od_parallel(
        et0, smas, incs, omegas, n_orbit=n_orbit, max_workers=10
    )

    # save results
    np.save("data/gridsearch_od/gridsearch_od_results.npy", results)

In [ ]:
np.save("data/gridsearch_od/gridsearch_od_results.npy", results)

In [ ]:
# 2d contour plot (sma vs raan)
validx = 3  # 3: pos 95, 7: pos rms
target_inc = 40
inc_idx = np.argmin(np.abs(incs - np.deg2rad(target_inc)))
with_text = False  # whether to print the text of coverage

fig = plt.figure(figsize=(10, 6))
omegad = np.rad2deg(omegas)
ogrid, agrid = np.meshgrid(omegad, smas / 1e3)  # [m] -> [km]

cf = plt.contourf(
    omegad,
    smas / 1000,
    results[:, inc_idx, :, validx],
    cmap="viridis",
    levels=50,
    alpha=0.3,
)
# add colorbar
cbar = plt.colorbar(cf)
cbar.set_label("Position error at 95% confidence [m]", fontsize=14, fontweight="bold")
plt.scatter(ogrid, agrid, c=results[:, inc_idx, :, validx], cmap="viridis")
# print the text of coverage
for i in range(len(smas)):
    for j in range(len(omegad)):
        eps_h = 200
        eps_i = 0.5
        if results[i, inc_idx, j, validx] > 0 and with_text:
            plt.text(
                omegad[j] + eps_i,
                smas[i] / 1000 + eps_h,
                f"{results[i, inc_idx, j, validx]:.2f}",
                ha="center",
                va="center",
                fontsize=9,
            )

# # plot the minimum altitude point
# plt.scatter(max_cov_incs, [min_alt + pnt.R_MOON] * len(max_cov_incs), c="red", marker="x", label="Minimum altitude")
# plt.colorbar()
plt.xlabel("RAAN [deg]", fontsize=14, fontweight="bold")
plt.ylabel("Semi-major axis [km]", fontsize=14, fontweight="bold")

# set minor grid to hlist and ilist
plt.grid()
plt.xticks(omegad)
plt.yticks(smas / 1000)
plt.ylim(pnt.R_MOON / 1000, smas[-1] / 1000 + 50)
plt.xlim(omegad[0] - 0.5, omegad[-1] + 0.5)
plt.savefig("figs/od/od_gridsearch_sma_omega_inc" + str(target_inc) + ".pdf", dpi=300)

In [ ]:
# 2d contour plot (sma vs inc)
validx = 3  # 7: rms 3: 95%
target_omega = 75  # deg
with_text = False

omega_idx = np.argmin(np.abs(omegas - np.deg2rad(target_omega)))
fig = plt.figure(figsize=(8, 5))
incd = np.rad2deg(incs)
igrid, agrid = np.meshgrid(incd, smas / 1e3)  # [m] -> [km]
cf = plt.contourf(
    incd,
    smas / 1000,
    results[:, :, omega_idx, validx],
    cmap="viridis",
    levels=50,
    alpha=0.3,
)
# add colorbar
cbar = plt.colorbar(cf)
cbar.set_label("Position error at 95% confidence [m]", fontsize=14, fontweight="bold")
plt.scatter(igrid, agrid, c=results[:, :, omega_idx, validx], cmap="viridis")
# print the text of coverage
for i in range(len(smas)):
    for j in range(len(incd)):
        eps_h = 200
        eps_i = 1.0
        if results[i, j, omega_idx, validx] > 0 and with_text:
            plt.text(
                incd[j] + eps_i,
                smas[i] / 1000 + eps_h,
                f"{results[i, j, omega_idx, validx]:.2f}",
                ha="center",
                va="center",
                fontsize=12,
            )
# # plot the minimum altitude point
# plt.scatter(max_cov_incs, [min_alt + pnt.R_MOON] * len(max_cov_incs), c="red", marker="x", label="Minimum altitude")
# plt.colorbar()
plt.xlabel("Inclination [deg]", fontsize=14, fontweight="bold")
plt.ylabel("Semi-major axis [km]", fontsize=14, fontweight="bold")
# set minor grid to hlist and ilist
plt.grid()
plt.xticks(incd)
plt.yticks(smas / 1000)
plt.ylim(3800, smas[-1] / 1000 + 200)
plt.xlim(incd[0] - 0.5, incd[-1] + 0.5)
plt.tight_layout()
plt.savefig(
    "figs/od/od_gridsearch_sma_inc_omega" + str(int(target_omega)) + ".pdf", dpi=300
)
plt.show()

## Run Single Case for Verification

In [ ]:
sma = 6000e3

sma_24hr = ((24 * 3600 / (2 * np.pi)) ** 2 * pnt.GM_MOON) ** (1 / 3)
sma = sma_24hr

inc = np.deg2rad(45)
Omega = np.deg2rad(165)  # 75, 165
ecc = np.sqrt(1 - 5 / 3 * np.cos(inc) ** 2)
peri_h = sma * (1 - ecc) - pnt.R_MOON
n_sat = len(omegas)
w = np.deg2rad(90)
M = np.deg2rad(0)
n_orbit = 4
stats_ratio = (n_orbit - 1) / n_orbit  # ratio of the stats period

T_orbit = 2 * np.pi * np.sqrt(sma**3 / pnt.GM_MOON)  # seconds
dt_od = 60  # OD interval in seconds
n_od = int(T_orbit / dt_od)  # number of OD
tspan = np.linspace(0, n_orbit * T_orbit, n_od + 1)  # 6 samples per hour
et = et0 + tspan  # convert to TAI

coe = np.array([sma, ecc, inc, Omega, w, M])
# initial states
rv0_mci = np.zeros((1, 6))  # initial states in MOON_CI frame
rv0_op = pnt.classical_to_cart(coe, pnt.GM_MOON)
rv0_mci = pnt.convert_frame(et0, rv0_op, pnt.MOON_OP, pnt.MOON_CI).reshape(1, 6)

# dynamics propagation
x_sat, stm_sat = propagate_sats_with_stm(rv0_mci, et, dynamics=dyn)

# measurement generation
elev, vis_moon, y, H = get_range_and_rate(
    et,
    odsim.pos_gs,
    x_sat,
    get_H=True,
    add_noise=True,
    sigma_range=1e-3,
    sigma_rangerate=1e-7,
)

# result dict
res = {
    "x_sat": x_sat,
    "stm_sat": stm_sat,
    "elev": elev,
    "vis_moon": vis_moon,
    "y": y,
    "H": H,
    "coe": coe,
}

# odsim
sigma_sat_mci, sigma_sat_rtn = odsim.run_sim(tspan, rv0_mci, dynamics=dyn, res=res)

print("shapes:")
print(sigma_sat_mci.shape)
print(sigma_sat_rtn.shape)
print(y.shape)
print(elev.shape)
print(tspan.shape)

In [ ]:
from src.odsim import fit_od_error

fit_params = fit_od_error(
    tspan,
    T_orbit,
    sigma_sat_rtn,
    stats_ratio=stats_ratio,
    debug=True,
    figname="figs/od/od_error_fit_sma_{0:.0f}_inc_{1:.0f}_Omega_{2:.0f}.pdf".format(
        sma / 1e3, int(np.rad2deg(inc)), int(np.rad2deg(Omega))
    ),
)
print(fit_params)

In [ ]:
from src.odsim import plot_od_result

figname = "figs/od/od_result_{0:.0f}_{1:.0f}_{2:.0f}.pdf".format(
    sma / 1e3, np.rad2deg(inc), np.rad2deg(Omega)
)
plot_od_result(
    tspan,
    elev,
    vis_moon,
    sigma_sat_mci,
    sigma_sat_rtn,
    elev_mask_deg=10,
    stats_ratio=3 / 4,
    use_logy=False,
    figname=figname,
)

In [ ]:
import plotly.graph_objects as go

w = np.deg2rad(90)
M = np.deg2rad(0)

# coes0 = np.array([12000, ecc, np.deg2rad(65), 0, w, M])
sma = 6000e3
inc = np.deg2rad(45)
ecc = np.sqrt(1 - 5 / 3 * np.cos(inc) ** 2)
print("Eccentricity:", ecc)
Omega1deg = 70
Omega2deg = 165
coes1 = np.array([sma, ecc, inc, np.rad2deg(Omega1deg), w, M])
coes2 = np.array([sma, ecc, inc, np.deg2rad(Omega2deg), w, M])

# propagate satellites
coe = np.vstack([coes1, coes2])
nsat = coe.shape[0]

# initial states
rv0_mci = np.zeros((n_sat, 6))  # initial states in MOON_CI frame
rv0_op = pnt.classical_to_cart(coe, pnt.GM_MOON)
rv0_mci = pnt.convert_frame(et0, rv0_op, pnt.MOON_OP, pnt.MOON_CI)

T_orbit = 2 * np.pi * np.sqrt(sma**3 / pnt.GM_MOON)  # seconds
tspan = np.linspace(0, n_orbit * T_orbit, n_orbit * 100)  # 6 samples per hour
et = et0 + tspan

# dynamics propagation
x_sat, stm_sat = propagate_sats_with_stm(rv0_mci, et, dynamics=dyn)

# moon to earth vector
m2e_mci = pnt.get_body_pos_vel(et0, pnt.MOON, pnt.EARTH, frame=pnt.MOON_CI)
print("Moon to Earth vector (km):", m2e_mci[:3])

# plot constellation
fig = go.Figure()
pnt.plot.plot_orbits(fig, x_sat[0], color="blue")
pnt.plot.plot_orbits(fig, x_sat[1], color="green")
# pnt.plot.plot_orbits(fig, x_sat[2], color='red')
pnt.plot.plot_body(
    fig,
    pnt.MOON,
    size_factor=2,
    alpha=0.5,
)
pnt.plot.plot_arrow3(
    fig, np.zeros(3), m2e_mci[:3] / 100, length=1.5, tip=0.5, color="black"
)
pnt.plot.set_view(fig, 10, 20, 2.8)
fig.update_layout(showlegend=True, width=400, height=400)
fig.write_image(
    "figs/od/constellation_2sat_Omega_{2:.0f}_{3:.0f}.pdf".format(
        sma / 1e3, np.rad2deg(inc), Omega1deg, Omega2deg
    ),
    scale=3,
)
fig.show()